# Testing accuracy of cellprofiler feature extraction for cell size - seeing which method is best

In [ ]:
import os
import numpy as np
import pandas as pd
import sqlite3
#plotting
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from scipy.stats import shapiro
import re
from scipy import stats

from helpers import *
from plate_preprocessing import *
from mitolyso_plot_functions import *

In [ ]:
## Import your csv files
csvpath = '/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs/'
filename = 'total_combined_cell.csv'
combined_cell_df_mitolyso = pd.read_csv(os.path.join(csvpath, filename))
#filter_df = cell_filters(combined_cell_df_mitolyso)
#display(combined_cell_df_mitolyso.shape)

stitched_path = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/stitching/segmentation_testset"
stitched_csv = "stitched_test_data.csv"
stitched_cells_df = pd.read_csv(Path.join(stitched_path,stitched_csv))

feature_meas = "Cell_AreaShape_Area"

In [ ]:
combined_cell_df_mitolyso = combined_cell_df[combined_cell_df['Staining'].str.startswith("LAMP1-488 + MitoRed")]
min_x = 0
min_y = 0
max_x = combined_cell_df_mitolyso["Image_Width_DAPI"][0] # get the max x and y resolutions
max_y = combined_cell_df_mitolyso["Image_Height_DAPI"][0]

combined_cell_df_mitolyso_borders_excluded = exclude_borders(combined_cell_df_mitolyso, min_x, min_y, max_x, max_y, prefix="Cell_")
combined_cell_df_mitolyso_borders_excluded.to_csv(os.path.join(outpath, "total_combined_cell_borders_excluded.csv"), index=False)
combined_cell_df_mitolyso_borders_excluded.describe().to_csv(os.path.join(outpath, "total_combined_cell_borders_excluded_stats.csv"))
combined_cell_df_mitolyso_borders_excluded.groupby("AllGroups")["Cell_AreaShape_Area"].describe().to_csv(os.path.join(outpath, "total_combined_cell_borders_excluded_passage_group_stats.csv"))

## Prep the stitched cell dataframe

In [ ]:
#rescale area to the original image by multiplying by the inverse of the rescale factor squared, e.g. for a factor of 1/4 then I can multipy by (4^2)=16
#original_area = measured_area * (1 / rescale_factor**2)

def find_replicate(path):
    replicate_pattern = r"R(\d{1})"  # Matches "RX" where X is the replicate number (placeholder for now)
    match = re.search(replicate_pattern, path)
    if match:
        replicate = int(match.group(1))
    else:
        replicate = None 
    return replicate

def find_row_col(well_code):
    rowcol_pattern = r"r(\d{1,2})c(\d{1,2})"  # Matches "RX" where X is the replicate number (placeholder for now)
    match = re.search(rowcol_pattern, well_code)
    if match:
        row_metadata = int(match.group(1))
        col_metadata = int(match.group(2))
    else:
        row_metadata = None 
        col_metadata = None
    return row_metadata,col_metadata

def prepare_stitched_cells_df(stitched_cells_df, feature_meas = "Cell_AreaShape_Area"):
    stitched_cells_df["Replicate_Number"] = stitched_cells_df.apply(
            lambda row: find_replicate(
                path=row["Path"]
            ), axis=1)
    stitched_cells_df[["Metadata_WellRow","Metadata_WellColumn"]] = stitched_cells_df.apply(
            lambda row: find_row_col(
                well_code=row["Well_id"]
            ), axis=1, result_type="expand")
    #rescale the area by a factor of 16 (inverse of 0.25^2)
    stitched_cells_df["Cell_AreaShape_Area"] = stitched_cells_df.apply(lambda x: x["area"]*16, axis=1)
    display(stitched_cells_df)

    unique_replicates = stitched_cells_df["Replicate_Number"].unique()
    metadata_sliced_dfs = [] #basicalyl split into 3 and rejoin them
    for i,rep in enumerate(unique_replicates):
        if rep == 5:
            map_file = "/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250328_rep05_metadata/map.csv"
        elif rep == 6:
            map_file = "/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250410_rep06_metadata/map.csv"
        elif rep == 7:
            map_file="/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250501_rep07_metadata/map.csv"
        if os.path.exists(map_file):
            platemap_df = pd.read_csv(map_file)
            platemap_df = platemap_df.drop_duplicates(subset=['Metadata_WellRow', 'Metadata_WellColumn']) #drop the dupes here, its ok bcus we only need the passage num
            platemap_df['Metadata_WellRow'] = platemap_df['Metadata_WellRow'].astype(int)
            platemap_df['Metadata_WellColumn'] = platemap_df['Metadata_WellColumn'].astype(int)

            #join the metadata and add to a list to join
            stitched_cells_df_prefilter = stitched_cells_df.merge(platemap_df, on=['Metadata_WellRow', 'Metadata_WellColumn'], how="left")
            stitched_cells_df_filter = stitched_cells_df_prefilter[stitched_cells_df_prefilter["Replicate_Number"]==rep]
            #display(stitched_cells_df_filter)
            metadata_sliced_dfs.append(stitched_cells_df_filter)
    #display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
    stitched_cells_df = pd.concat(metadata_sliced_dfs)
    stitched_cells_df["Passage Group"] = stitched_cells_df['PassageNumber'].apply(passage_group)
    stitched_cells_df["AllGroups"] = add_drug_to_group(stitched_cells_df, "Passage Group", "Drug")
    display(stitched_cells_df)

In [ ]:
prepare_stitched_cells_df(stitched_cells_df)

group_avg_df = average_groups_by_plate(combined_cell_df_mitolyso, x_value='AllGroups', y_value="Cell_AreaShape_Area", replicates='Replicate_Number')
group_avg_df_borders_excluded = average_groups_by_plate(combined_cell_df_mitolyso_borders_excluded, x_value='AllGroups', y_value="Cell_AreaShape_Area", replicates='Replicate_Number')
group_avg_df = apply_shapiro_wilk_test_to_df(group_avg_df,feature_meas)
group_avg_df_borders_excluded = apply_shapiro_wilk_test_to_df(group_avg_df_borders_excluded,feature_meas)
display(group_avg_df_borders_excluded)


## code to make the side-by-side comparison plots

In [ ]:
def superplot_for_area_threshold_comparisons(data_df_1, group_avg_df_1, data_df_2, group_avg_df_2, x_value="AllGroups", y_value="Cell_AreaShape_Area", replicate_col_name = "Replicate_Number", csv_dir="", xtitle=None, ytitle=None, order = None, title2="Excluding Cells Touching Borders", annotate = False):
    """Make two side-by-side superplots to compare area between different conditions
    Args:
        data_df_1 (_type_): _description_
        group_avg_df_1 (_type_): _description_
        data_df_2 (_type_): _description_
        group_avg_df_2 (_type_): _description_
        x_value (str, optional): _description_. Defaults to "AllGroups".
        y_value (str, optional): _description_. Defaults to "Cell_AreaShape_Area".
        replicate_col_name (str, optional): _description_. Defaults to "Replicate_Number".
        csv_dir (str, optional): _description_. Defaults to "".
        xtitle (_type_, optional): _description_. Defaults to None.
        ytitle (_type_, optional): _description_. Defaults to None.
    """
    import matplotlib.lines as mlines
    from statannotations.Annotator import Annotator
    from statannotations.stats.StatTest import StatTest
    if order == None:
        order = get_all_group_order()
    pairs = getpairs(data_df_2,x_value,order=order)
    print(pairs)
        
    fig, axes = plt.subplots(1, 2, figsize=(30, 10), sharey=True, sharex=False)
    #plt.style.use("ggplot")
    sns.set_context("talk", font_scale=1.2)
    sns.set_theme(style="whitegrid")
    
    #First subplot: Full Dataset
    # sns.stripplot(
    #     data=data_df_1, 
    #     x=x_value, y=y_value, hue=x_value,
    #     palette="Set2", 
    #     order=order,
    #     ax=axes[0],
    # )
    sns.violinplot(
        data=data_df_1, 
        x=x_value, y=y_value, #hue=x_value,
        #palette="Set2", 
        split=True, #using split violin plots - only one side, basically looks like a histogram
        inner ="quart",
        color="gainsboro",
        width=0.9, 
        linewidth=1.5,
        order=order,
        ax=axes[0],
    )
    sns.swarmplot(
        data=group_avg_df_1, x=x_value, y=y_value,
        hue=replicate_col_name, 
        order=order, 
        palette='pastel',
        size=12, 
        edgecolor="k", 
        linewidth=1, 
        dodge=False, 
        ax=axes[0]
    )
    #draw a boxplot to show the mean line
    sns.boxplot(data=group_avg_df_1, x=x_value, y=y_value,
        showmeans=True,
        meanline=True,
        meanprops={'color': 'dimgray', 'ls': '-', 'lw': 2.5},
        medianprops={'visible': False},
        whiskerprops={'visible': False},
        zorder=2,
        showfliers=False,
        showbox=False,
        showcaps=False,
        ax=axes[0])
    axes[0].set_title("Full Dataset")
    
        # axes[0].text(
        #     x=row[x_value], 
        #     y=row[y_value], 
        #     s=str(row["Shapiro_normality"]), 
        #     color="black", 
        #     fontsize=10,
        #     ha="center"
        # )
    group_avg_df_1_pivot = average_groups_pivot(group_avg_df_1, x_value, y_value, replicate_col_name)
    if annotate:
        annotator = Annotator(axes[0], pairs, data=group_avg_df_1, order=order) 
        annotator.configure(test='Kruskal', 
                            text_format='star', 
                            #pvalue_format = 'simple',
                            loc='inside', 
                            hide_non_significant = False,
                            color = 'black',
                            verbose = 2)
        annotator.apply_and_annotate()
    #add an annotation to legend if there is normality
    
    unique_replicates = group_avg_df_1[replicate_col_name].unique()
    L = plt.legend()
    custom_labels = []
    for rep in unique_replicates:
        label = str(rep)
        shapiro_val = group_avg_df_1[group_avg_df_1[replicate_col_name] == rep]["Shapiro_normality"].iloc[0]
        if shapiro_val:
            label += " (normal)"
        custom_labels.append(label)

    # Create custom legend handles (using the same colors as swarmplot)
    palette = sns.color_palette('pastel', n_colors=len(unique_replicates))
    handles = [
        mlines.Line2D([], [], color=palette[i], 
                      marker='o', 
                      linestyle='None', 
                      markersize=12, 
                      markeredgecolor='black', 
                      label=custom_labels[i])
        for i in range(len(unique_replicates))]
    axes[0].legend_.set_title("Replicate")
    axes[0].legend(handles=handles, title="Replicate", loc="best")
    #axes[0].legend_.remove()

    # Second subplot: Excluding Borders
    sns.violinplot(
        data=data_df_2, 
        x=x_value, y=y_value,# hue=x_value,
        #palette="Set2", 
        split=True,
        inner ="quart",
        color="gainsboro",
        width=0.9, 
        linewidth=1.5,
        order=order,
        ax=axes[1],)
    sns.swarmplot(
        data=group_avg_df_2, x=x_value, y=y_value,
        hue=replicate_col_name, 
        order=order, 
        palette='pastel',
        size=12, 
        edgecolor="k", 
        linewidth=1, 
        dodge=False, 
        ax=axes[1]
    )
    sns.boxplot(data=group_avg_df_2, x=x_value, y=y_value,
        showmeans=True,
        meanline=True,
        meanprops={'color': 'dimgray', 'ls': '-', 'lw': 2.5},
        medianprops={'visible': False},
        whiskerprops={'visible': False},
        zorder=2,
        showfliers=False,
        showbox=False,
        showcaps=False,
        ax=axes[1])
    axes[1].set_title(title2)
    axes[1].legend_.remove()
    if ytitle is not None:
        axes[0].set_ylabel(ytitle)  
    if xtitle is not None:
        axes[0].set_xlabel(xtitle)  
        axes[1].set_xlabel(xtitle) 
    
    group_avg_df_2_pivot = average_groups_pivot(group_avg_df_2, x_value, y_value, replicate_col_name)
    unique_replicates = group_avg_df_2[replicate_col_name].unique()
    L2 = plt.legend()
    custom_labels = []
    for rep in unique_replicates:
        label = str(rep)
        shapiro_val = group_avg_df_2[group_avg_df_2[replicate_col_name] == rep]["Shapiro_normality"].iloc[0]
        if shapiro_val:
            label += " (normal)"
        custom_labels.append(label)

    # Create custom legend handles (using the same colors as swarmplot)
    palette = sns.color_palette('pastel', n_colors=len(unique_replicates))
    handles = [
        mlines.Line2D([], [], color=palette[i], 
                      marker='o', 
                      linestyle='None', 
                      markersize=12, 
                      markeredgecolor='black', 
                      label=custom_labels[i])
        for i in range(len(unique_replicates))]
    axes[1].legend_.set_title("Replicate")
    axes[1].legend(handles=handles, title="Replicate", loc="best")
    
    if annotate:
        annotator = Annotator(axes[1], pairs, data=group_avg_df_2, order=order) 
        annotator.configure(test='Kruskal', 
                            text_format='star', 
                            #pvalue_format = 'simple',
                            loc='inside', 
                            hide_non_significant = False,
                            color = 'black',
                            verbose = 2)
        annotator.apply_and_annotate()
    
    plt.tight_layout()
    plt.savefig(os.path.join(csv_dir, f"combined_cellsize_boxplots_{title2}.png"))
    plt.show()
    
    
    group_avg_df_1_pivot.to_csv("area_pivot.csv") #can plop this into graphpad and see what it tells me
    group_avg_df_2_pivot.to_csv(f"area_pivot_{title2}.csv")

    
superplot_for_area_threshold_comparisons(combined_cell_df_mitolyso, group_avg_df, combined_cell_df_mitolyso_borders_excluded, group_avg_df_borders_excluded, x_value="AllGroups", y_value="Cell_AreaShape_Area", csv_dir=csv_dir, xtitle = "Passage Groups", ytitle="Cell Area")


## Make plots and csvs for the comparison btwn regular images and the stitched images

In [ ]:
stitch_group_avg_df = average_groups_by_plate(stitched_cells_df, x_value='AllGroups', y_value="Cell_AreaShape_Area", replicates='Replicate_Number')
stitch_group_avg_df = apply_shapiro_wilk_test_to_df(stitch_group_avg_df,"Cell_AreaShape_Area")

combined_cell_df_mitolyso["Metadata_Well"] = combined_cell_df_mitolyso.apply(lambda x: well_namer(x["Metadata_WellRow"],x["Metadata_WellColumn"]), axis=1)
combined_cell_df_mitolyso_subset_extrawells = combined_cell_df_mitolyso[combined_cell_df_mitolyso["TimepointName"].isin(stitched_cells_df["TimepointName"])]
combined_cell_df_mitolyso_subset = combined_cell_df_mitolyso_subset_extrawells[combined_cell_df_mitolyso_subset_extrawells["Metadata_Well"].isin(stitched_cells_df["Metadata_Well"])]
display(combined_cell_df_mitolyso_subset)
#display(combined_cell_df_mitolyso_subset)
group_avg_df_subset = average_groups_by_plate(combined_cell_df_mitolyso_subset, x_value='AllGroups', y_value="Cell_AreaShape_Area", replicates='Replicate_Number')
group_avg_df_subset = apply_shapiro_wilk_test_to_df(group_avg_df_subset,"Cell_AreaShape_Area")
superplot_for_area_threshold_comparisons(combined_cell_df_mitolyso_subset, group_avg_df_subset, stitched_cells_df, stitch_group_avg_df, order=["P6-10","P17-19","P29+","Doxo"], title2="From Stitched Images", annotate = False)

outpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs"
stitched_cells_df.to_csv(os.path.join(outpath, "stitched_cells.csv"), index=False)
stitched_cells_df.describe().to_csv(os.path.join(outpath, "stitched_cell_stats.csv"))
stitched_cells_df.groupby("AllGroups")["Cell_AreaShape_Area"].describe().to_csv(os.path.join(outpath, "stitched_cell_passage_group_stats.csv"))
